In [1]:
import duckdb

CSV = "../data/raw/PS_20174392719_1491204439457_log.csv"

# Peek at the structure — instant, doesn't load into memory
duckdb.sql(f"SELECT * FROM '{CSV}' LIMIT 10").show()

┌───────┬──────────┬──────────┬─────────────┬───────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────┬────────────────┐
│ step  │   type   │  amount  │  nameOrig   │ oldbalanceOrg │ newbalanceOrig │  nameDest   │ oldbalanceDest │ newbalanceDest │ isFraud │ isFlaggedFraud │
│ int64 │ varchar  │  double  │   varchar   │    double     │     double     │   varchar   │     double     │     double     │  int64  │     int64      │
├───────┼──────────┼──────────┼─────────────┼───────────────┼────────────────┼─────────────┼────────────────┼────────────────┼─────────┼────────────────┤
│     1 │ PAYMENT  │  9839.64 │ C1231006815 │      170136.0 │      160296.36 │ M1979787155 │            0.0 │            0.0 │       0 │              0 │
│     1 │ PAYMENT  │  1864.28 │ C1666544295 │       21249.0 │       19384.72 │ M2044282225 │            0.0 │            0.0 │       0 │              0 │
│     1 │ TRANSFER │    181.0 │ C1305486145 │         181.0 │            0.0

In [2]:
# Row count and column types
duckdb.sql(f"DESCRIBE SELECT * FROM '{CSV}'").show()
duckdb.sql(f"SELECT COUNT(*) AS rows FROM '{CSV}'").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ step           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ type           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ amount         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ nameOrig       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ oldbalanceOrg  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ newbalanceOrig │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ nameDest       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ oldbalanceDest │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ newbalanceDest │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ isFraud        │ BIGINT      │ YES     │ NULL    

In [3]:
# Transaction types and fraud distribution
duckdb.sql(f"""
    SELECT
        type,
        COUNT(*)                          AS n_transactions,
        SUM(isFraud)                      AS n_fraud,
        ROUND(100.0 * SUM(isFraud) / COUNT(*), 4) AS fraud_pct
    FROM '{CSV}'
    GROUP BY type
    ORDER BY n_transactions DESC
""").show()

┌──────────┬────────────────┬─────────┬───────────┐
│   type   │ n_transactions │ n_fraud │ fraud_pct │
│ varchar  │     int64      │ int128  │  double   │
├──────────┼────────────────┼─────────┼───────────┤
│ CASH_OUT │        2237500 │    4116 │     0.184 │
│ PAYMENT  │        2151495 │       0 │       0.0 │
│ CASH_IN  │        1399284 │       0 │       0.0 │
│ TRANSFER │         532909 │    4097 │    0.7688 │
│ DEBIT    │          41432 │       0 │       0.0 │
└──────────┴────────────────┴─────────┴───────────┘



In [4]:
# The merchant pattern
duckdb.sql(f"""
    SELECT
        CASE WHEN nameDest LIKE 'M%' THEN 'merchant' ELSE 'customer' END AS dest_type,
        COUNT(*) AS n,
        AVG(oldbalanceDest) AS avg_old_balance
    FROM '{CSV}'
    GROUP BY 1
""").show()

┌───────────┬─────────┬────────────────────┐
│ dest_type │    n    │  avg_old_balance   │
│  varchar  │  int64  │       double       │
├───────────┼─────────┼────────────────────┤
│ merchant  │ 2151495 │                0.0 │
│ customer  │ 4211125 │ 1663058.3127860746 │
└───────────┴─────────┴────────────────────┘



In [5]:
# The step column: 1 step = 1 hour, 744 steps = 30 days
duckdb.sql(f"SELECT MIN(step), MAX(step) FROM '{CSV}'").show()

┌───────────┬───────────┐
│ min(step) │ max(step) │
│   int64   │   int64   │
├───────────┼───────────┤
│         1 │       743 │
└───────────┴───────────┘



### Finding: fraud is confined to two transaction types

Fraud appears only in TRANSFER (0.77%) and CASH_OUT (0.18%). The other
three types contain zero fraud. Absolute counts are near-identical
(4,097 vs 4,116), consistent with a transfer-then-cash-out laundering
pattern. This narrows the detection problem to ~2.77M of 6.36M rows.


## Key Findings

- **Fraud occurs only in:** `TRANSFER` and `CASH_OUT` transaction types.

- **Merchant identification:**
  - Destination account names starting with **`M`** represent merchants.

- **Merchant balance information:**
  - Merchants have an average old balance (`oldbalanceDest`) of **0.0**.
  - This suggests that **merchant accounts do not have balance information recorded**.

- **Merchant destinations vs. PAYMENT transactions:**
  - Merchant destinations: **2,151,495**
  - `PAYMENT` transactions: **2,151,495**
  - **Observation:** These counts match exactly, indicating that every `PAYMENT` transaction is sent to a merchant.

- **Fraud distribution:**
  - `TRANSFER`: **4,097** fraudulent transactions
  - `CASH_OUT`: **4,116** fraudulent transactions
  - **Observation:** Both transaction types have almost identical numbers of fraudulent transactions.